## Импорт библиотек

In [1]:
import pandas as pd
import string
import re
import nltk
import pymorphy3

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
df = pd.read_csv('habr.csv')

In [3]:
df.head()

,title,description,date,tags,rating
0,"Cocoapods, Carthage, SPM: как выбрать менеджер...",117.94 Рейтинг red_mad_robot №1 в разработке ...,6 часов назад,"Хабы: Блог компании red_mad_robot , Разработ...",117.94
1,Deutsche Telekom и Perplexity объявили о новом...,"Еще до начала MWC в Барселоне было очевидно, ...",1 час назад,"Хабы: Искусственный интеллект, Смартфоны",4.40
2,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y, Информационная б...",71.07
3,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y , Информационная...",71.07
4,Быстрое начало работы с Gitlab CI/CD: пайплайн...,4.29 Оценка 280.79 Рейтинг Southbridge Обеспе...,2 часа назад,"Хабы: Блог компании Southbridge , Тестирован...",280.79


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        15 non-null     object 
 1   description  15 non-null     object 
 2   date         15 non-null     object 
 3   tags         15 non-null     object 
 4   rating       15 non-null     float64
dtypes: float64(1), object(4)
memory usage: 732.0+ bytes


## Предварительная обработка данных

In [5]:
import re

def remove_english_words(text):
    # Удаляет слова, состоящие только из английских букв (включая сокращения)
    return re.sub(r'\b[a-zA-Z]+\b', '', text)

def remove_punctuation(text): 
    return "".join([ch if ch not in string.punctuation else ' ' for ch in text])

def remove_numbers(text): 
    return ''.join([i if not i.isdigit() else ' ' for i in text])

def remove_multiple_spaces(text): 
    return re.sub(r'\s+', ' ', text, flags=re.I)

st = '❯\xa0—«»'
def remove_othersymbol(text):
    return ''.join([ch if ch not in st else ' ' for ch in text])

In [6]:
df['prep_text'] = [remove_english_words(remove_multiple_spaces(remove_numbers(remove_othersymbol(remove_punctuation(text.lower()))))) for text in df['description']]

In [7]:
russian_stopwords = stopwords.words("russian")
russian_stopwords.extend(['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 
    'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 
    'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 
    'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 
    'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'жизнь', 
    'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 
    'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'другой', 
    'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 
    'моя', 'свою', 'этой', 'перед', 'иногда', 'лучше', 'чуть', 'том', 'нельзя', 'такой', 'им', 'более', 'всегда', 'конечно', 'всю', 
    'между', 'это', 'пока', 'об', 'часто', 'теперь', 'слишком', 'либо','хабр', 'хабрахабр', 'habr', 'комментарий', 'комментарии', 'статья', 'статьи', 'автор', 'пользователь', 'пост', 'блог',
    'читать', 'прочитать', 'написать', 'написал', 'подробнее', 'рейтинг', 'плюс', 'минус', 'карма', 'хаб', 'хабы',
    'git', 'github', 'пример', 'код', 'кода', 'коде', 'коду', 'кодом', 'python', 'javascript', 'программа', 'проект', 
    'использовать', 'использование', 'файл', 'файлы', 'сделать', 'создать', 'решение', 'задача', 'задачи', 'работа', 
    'примеры', 'документация', 'класс', 'функция', 'метод', 'сервер', 'клиент', 'данные', 'данных', 'база', 'базы',
    'http', 'https', 'www', 'com', 'ru', 'ссылка', 'img', 'src', 'div', 'класса', 'alt', 'nbsp', 'quot', 'lt', 'gt', 
    'является', 'являются', 'можно', 'найти', 'стоит', 'хочу', 'хотел', 'кажется', 'получить', 'помощью',
    'сегодня', 'вчера', 'неделя', 'месяц', 'год', 'года', 'лет', 'время', 'раз', 'версия'])

In [8]:
def tokenize(text):
    t = word_tokenize(text)
    tokens = [token for token in t if token not in russian_stopwords]
    text = " ".join(tokens)
    return text

In [9]:
df['tokenize_text'] = [tokenize(text) for text in df['prep_text']]

In [10]:
df.head()

,title,description,date,tags,rating,prep_text,tokenize_text
0,"Cocoapods, Carthage, SPM: как выбрать менеджер...",117.94 Рейтинг red_mad_robot №1 в разработке ...,6 часов назад,"Хабы: Блог компании red_mad_robot , Разработ...",117.94,рейтинг № в разработке цифровых решений дл...,№ разработке цифровых решений бизнеса часов на...
1,Deutsche Telekom и Perplexity объявили о новом...,"Еще до начала MWC в Барселоне было очевидно, ...",1 час назад,"Хабы: Искусственный интеллект, Смартфоны",4.40,еще до начала в барселоне было очевидно что ...,начала барселоне очевидно хотя оператор предст...
2,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y, Информационная б...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...
3,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y , Информационная...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...
4,Быстрое начало работы с Gitlab CI/CD: пайплайн...,4.29 Оценка 280.79 Рейтинг Southbridge Обеспе...,2 часа назад,"Хабы: Блог компании Southbridge , Тестирован...",280.79,оценка рейтинг обеспечиваем стабильную работ...,оценка обеспечиваем стабильную работу проектов...


In [11]:
stemmer = SnowballStemmer("russian")

stem_list = []
for text in (df['tokenize_text']):
    try:
        tokens = word_tokenize(text)
        res = list()
        for word in tokens:
            p = stemmer.stem(word)
            res.append(p)
        text = " ".join(res)
        stem_list.append(text)
    except Exception as e:
        print(e)
        
df['text_stem'] = stem_list

In [12]:
df.head()

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem
0,"Cocoapods, Carthage, SPM: как выбрать менеджер...",117.94 Рейтинг red_mad_robot №1 в разработке ...,6 часов назад,"Хабы: Блог компании red_mad_robot , Разработ...",117.94,рейтинг № в разработке цифровых решений дл...,№ разработке цифровых решений бизнеса часов на...,№ разработк цифров решен бизнес час назад выбр...
1,Deutsche Telekom и Perplexity объявили о новом...,"Еще до начала MWC в Барселоне было очевидно, ...",1 час назад,"Хабы: Искусственный интеллект, Смартфоны",4.40,еще до начала в барселоне было очевидно что ...,начала барселоне очевидно хотя оператор предст...,нача барселон очевидн хот оператор представ ам...
2,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y, Информационная б...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом ‑ак...
3,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y , Информационная...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом акк...
4,Быстрое начало работы с Gitlab CI/CD: пайплайн...,4.29 Оценка 280.79 Рейтинг Southbridge Обеспе...,2 часа назад,"Хабы: Блог компании Southbridge , Тестирован...",280.79,оценка рейтинг обеспечиваем стабильную работ...,оценка обеспечиваем стабильную работу проектов...,оценк обеспечива стабильн работ проект оригина...


In [13]:
morph = pymorphy3.MorphAnalyzer(lang='ru')

In [14]:
%%time
lemm_texts_list = []
for text in (df['tokenize_text']):
    try:
        tokens = word_tokenize(text)
        res = list()
        for word in tokens:
            p = morph.parse(word)[0]
            res.append(p.normal_form)
        text = " ".join(res)
        lemm_texts_list.append(text)
    except Exception as e:
        print(e)
    
df['text_lemm'] = lemm_texts_list

CPU times: total: 4.02 s
Wall time: 4.21 s


In [15]:
df.head()

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm
0,"Cocoapods, Carthage, SPM: как выбрать менеджер...",117.94 Рейтинг red_mad_robot №1 в разработке ...,6 часов назад,"Хабы: Блог компании red_mad_robot , Разработ...",117.94,рейтинг № в разработке цифровых решений дл...,№ разработке цифровых решений бизнеса часов на...,№ разработк цифров решен бизнес час назад выбр...,№ разработка цифровой решение бизнес час назад...
1,Deutsche Telekom и Perplexity объявили о новом...,"Еще до начала MWC в Барселоне было очевидно, ...",1 час назад,"Хабы: Искусственный интеллект, Смартфоны",4.40,еще до начала в барселоне было очевидно что ...,начала барселоне очевидно хотя оператор предст...,нача барселон очевидн хот оператор представ ам...,начало барселона очевидный хотя оператор предс...
2,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y, Информационная б...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом ‑ак...,корпоративный облачный провайдер оригинал взло...
3,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y , Информационная...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом акк...,корпоративный облачный провайдер оригинал взло...
4,Быстрое начало работы с Gitlab CI/CD: пайплайн...,4.29 Оценка 280.79 Рейтинг Southbridge Обеспе...,2 часа назад,"Хабы: Блог компании Southbridge , Тестирован...",280.79,оценка рейтинг обеспечиваем стабильную работ...,оценка обеспечиваем стабильную работу проектов...,оценк обеспечива стабильн работ проект оригина...,оценка обеспечивать стабильный работа проект о...


In [16]:
df['text_lemm'] = [tokenize(text) for text in df['text_lemm']]

In [17]:
df.head()

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm
0,"Cocoapods, Carthage, SPM: как выбрать менеджер...",117.94 Рейтинг red_mad_robot №1 в разработке ...,6 часов назад,"Хабы: Блог компании red_mad_robot , Разработ...",117.94,рейтинг № в разработке цифровых решений дл...,№ разработке цифровых решений бизнеса часов на...,№ разработк цифров решен бизнес час назад выбр...,№ разработка цифровой бизнес час назад выбрать...
1,Deutsche Telekom и Perplexity объявили о новом...,"Еще до начала MWC в Барселоне было очевидно, ...",1 час назад,"Хабы: Искусственный интеллект, Смартфоны",4.40,еще до начала в барселоне было очевидно что ...,начала барселоне очевидно хотя оператор предст...,нача барселон очевидн хот оператор представ ам...,начало барселона очевидный хотя оператор предс...
2,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y, Информационная б...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом ‑ак...,корпоративный облачный провайдер оригинал взло...
3,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y , Информационная...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом акк...,корпоративный облачный провайдер оригинал взло...
4,Быстрое начало работы с Gitlab CI/CD: пайплайн...,4.29 Оценка 280.79 Рейтинг Southbridge Обеспе...,2 часа назад,"Хабы: Блог компании Southbridge , Тестирован...",280.79,оценка рейтинг обеспечиваем стабильную работ...,оценка обеспечиваем стабильную работу проектов...,оценк обеспечива стабильн работ проект оригина...,оценка обеспечивать стабильный оригинал переве...


## Векторизация текстовых данных

In [18]:
tfidf_vectorizer = TfidfVectorizer(max_df=1, max_features=100000000,
                                 min_df=1, stop_words=russian_stopwords,
                                 ngram_range=(1,3))

In [19]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text_lemm'])

In [20]:
tfidf_matrix.shape

(15, 16843)

## Тематическое моделирование

In [21]:
from sklearn.decomposition import TruncatedSVD

In [22]:
# создание модели LSA
lsa_model = TruncatedSVD(n_components=5, random_state=0)
lsa_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: системный аналитик, новый процессор, альфа банк, альфа, трудовой, рассказывать строить карьера, плавать, софт искусственный, привычный, упоминать
Topic 1: смартфон, альфа, альфа банк, сеть сообщество обзор, сутки похожий сайт, песочница дата, песочница дата регистрация, спасибо внимание корпоративный, сайт вконтакте ещё, сайт ваш аккаунт
Topic 2: смартфон, ан, заря, ан заря, системный аналитик, сообщество обзор, песочница дата, внимание корпоративный, внимание корпоративный облачный, компания песочница дата
Topic 3: смартфон, системный аналитик, трудовой, цюрих, ггц, лондон, немат, сезон хабра, сезон, робот
Topic 4: трассировка, первопричина, системный аналитик, алерот, выявление, производительность приложение, анализ первопричина, популярный способ, аномалия, выявление аномалия


In [23]:
from sklearn.decomposition import NMF

In [24]:
# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: новый процессор, менеджер зависимость, упоминать, способный поддерживать, рассказывать строить карьера, плавать, привычный, сообщаться, строить карьера, скоро
Topic 1: датасаентист, заказчик, онлайн, бейкер, онлайн календарь, февраль онлайн календарь, календарь, февраль онлайн, навык, хватить
Topic 2: системный аналитик, трудовой, робот, сезон, сезон хабра, хотеть бесплатно, копия резюме, марвин ждать, марвин, екатеринбург системный аналитик
Topic 3: вконтакте публикация канал, публикация канал розыгрыш, разработка ра, разработка ра несмотря, разработка разработка хороший, персональный компьютер существовать, разработка хороший, администрирование разработка ра, разработка хороший сутки, паяльник компания
Topic 4: задание, лондон, цюрих, ггц, учёный, инстанс, непрерывный, сеть сообщесть, безопасность чтоть вроде, социальный сеть сообщесть


## Кластеризация

In [25]:
num_clusters = 5

# Метод к-средних - KMeans
from sklearn.cluster import KMeans
km = KMeans(n_clusters=num_clusters, random_state=0)

In [26]:
km.fit(tfidf_matrix)

  File "C:\Users\robhul\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\robhul\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\robhul\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\robhul\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


KMeans(n_clusters=5, random_state=0)

In [27]:
clusterkm = km.labels_.tolist()
df['cluster']= clusterkm

In [28]:
df.head()

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
0,"Cocoapods, Carthage, SPM: как выбрать менеджер...",117.94 Рейтинг red_mad_robot №1 в разработке ...,6 часов назад,"Хабы: Блог компании red_mad_robot , Разработ...",117.94,рейтинг № в разработке цифровых решений дл...,№ разработке цифровых решений бизнеса часов на...,№ разработк цифров решен бизнес час назад выбр...,№ разработка цифровой бизнес час назад выбрать...,1
1,Deutsche Telekom и Perplexity объявили о новом...,"Еще до начала MWC в Барселоне было очевидно, ...",1 час назад,"Хабы: Искусственный интеллект, Смартфоны",4.40,еще до начала в барселоне было очевидно что ...,начала барселоне очевидно хотя оператор предст...,нача барселон очевидн хот оператор представ ам...,начало барселона очевидный хотя оператор предс...,1
2,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y, Информационная б...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом ‑ак...,корпоративный облачный провайдер оригинал взло...,1
3,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y , Информационная...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом акк...,корпоративный облачный провайдер оригинал взло...,1
4,Быстрое начало работы с Gitlab CI/CD: пайплайн...,4.29 Оценка 280.79 Рейтинг Southbridge Обеспе...,2 часа назад,"Хабы: Блог компании Southbridge , Тестирован...",280.79,оценка рейтинг обеспечиваем стабильную работ...,оценка обеспечиваем стабильную работу проектов...,оценк обеспечива стабильн работ проект оригина...,оценка обеспечивать стабильный оригинал переве...,3


In [29]:
df['cluster'].value_counts()

cluster
1    11
3     1
0     1
4     1
2     1
Name: count, dtype: int64

In [30]:
df[df['cluster']==0]

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
8,История российской науки: напишем вместе? 1.3K,4.58 Оценка 353.18 Рейтинг Хабр Экосистема дл...,10 часов назад,"Хабы: Блог компании Хабр, Научно-популярное",353.18,оценка рейтинг хабр экосистема для развития л...,оценка экосистема развития людей вовлеченных ф...,оценк экосистем развит люд вовлечен феврал ден...,оценка экосистема развитие человек вовлечь фев...,0


In [31]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==0]['text_lemm'])
print('модель NMF:')
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i+1}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

модель NMF:
Topic 1: учёный, российский, первый, история, который, стать, наука, российский наука, знать, пирог
Topic 2: лишь, пора, заслуга, самый, самый засекретить учёный, экосистема развитие человек, горный инженер, назад история, кулибина, вовлечь
Topic 3: аэс самый засекретить, знать интересно знать, экранизация день, течение пройти подобный, анонимный дед мороз, первый европа атомный, разум помочь различный, николаевич корсак создатель, формат свободный желательно, сей пора
Topic 4: камень глазной, корпоративный медийный, любовь, описание приятный, булат павел петрович, разведка, электропочта информация сайт, исследовать природа южный, краткий, лента
Topic 5: урал вести геологический, привить любовь, узор, вид стать, травматизм, область ядерный физика, воздушный ткань несколько, дек поздравление загадка, отразить фио учёный, возможность оперировать


In [32]:
df[df['cluster']==1]

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
0,"Cocoapods, Carthage, SPM: как выбрать менеджер...",117.94 Рейтинг red_mad_robot №1 в разработке ...,6 часов назад,"Хабы: Блог компании red_mad_robot , Разработ...",117.94,рейтинг № в разработке цифровых решений дл...,№ разработке цифровых решений бизнеса часов на...,№ разработк цифров решен бизнес час назад выбр...,№ разработка цифровой бизнес час назад выбрать...,1
1,Deutsche Telekom и Perplexity объявили о новом...,"Еще до начала MWC в Барселоне было очевидно, ...",1 час назад,"Хабы: Искусственный интеллект, Смартфоны",4.40,еще до начала в барселоне было очевидно что ...,начала барселоне очевидно хотя оператор предст...,нача барселон очевидн хот оператор представ ам...,начало барселона очевидный хотя оператор предс...,1
2,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y, Информационная б...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом ‑ак...,корпоративный облачный провайдер оригинал взло...,1
3,OSINT & Hacking — как работает фишинг для нель...,71.07 Рейтинг Cloud4Y #1 Корпоративный облачн...,5 часов назад,"Хабы: Блог компании Cloud4Y , Информационная...",71.07,рейтинг корпоративный облачный провайдер ав...,корпоративный облачный провайдер оригинала взл...,корпоративн облачн провайдер оригина взлом акк...,корпоративный облачный провайдер оригинал взло...,1
5,Генеральный директор Mozilla покинула свой пост,"Митчелл Бейкер, гендиректор Mozilla с 2020 го...",12 минут назад,"Хабы: IT-компании, Управление персоналом, Кар...",145.10,митчелл бейкер гендиректор с года объявила ч...,митчелл бейкер гендиректор объявила покидает с...,митчелл бейкер гендиректор объяв покида сво во...,митчелла бейкер гендиректор объявить покидать ...,1
6,ИИ-агенты в Альфа-Банке: нейросети создают авт...,404.32 Рейтинг Альфа-Банк Лучший мобильный ба...,2 часа назад,"Хабы: Блог компании Альфа-Банк, Искусственный ...",404.32,рейтинг альфа банк лучший мобильный банк по в...,альфа банк лучший мобильный банк версии подпис...,альф банк лучш мобильн банк верс подписа альф ...,альфа банк хороший мобильный банк подписаться ...,1
7,"9 мин Инструменты наблюдаемости, о которых",2376.9 Рейтинг Автор оригинала: Lahiru Hewa...,4 часа назад,"Хабы: Блог компании , Open source , Хранени...",2376.90,рейтинг автор оригинала часа назад мин инст...,оригинала часа назад мин инструменты наблюдаем...,оригина час назад мин инструмент наблюдаем кот...,оригинал час назад мина инструмент наблюдаемос...,1
11,9 мин Как создать аппаратный эмулятор CD-ROM,2394.92 Рейтинг 20 мар в 12:00 9 мин Как ...,20 мар в 12:00,"Хабы: Блог компании , Системное администрир...",2394.92,рейтинг мар в мин как создать аппаратный эмул...,мар мин аппаратный эмулятор паяльника компании...,мар мин аппаратн эмулятор паяльник компан сист...,мара мина аппаратный эмулятор паяльник компани...,1
12,Может ли chatGPT забронировать столик в рестор...,Идея А почему бы не использовать возможност...,6 часов назад,"Хабы: Мессенджеры , Python , Искусственный и...",14.00,идея а почему бы не использовать возможности ...,идея почему возможности попросить делать напри...,иде поч возможн попрос дела например дава попр...,идея почему возможность попросить делать напри...,1
13,Новые утечки. Что мы знаем о выходе Windows 12,4.74 Оценка 414.35 Рейтинг getmatch Рассказыв...,4 часа назад,"Хабы: Блог компании getmatch , Разработка по...",414.35,оценка рейтинг рассказываем о том как строит...,оценка рассказываем строить карьеру весь интер...,оценк рассказыва стро карьер ве интернет готов...,оценка рассказывать строить карьера весь интер...,1


In [33]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==1]['text_lemm'])
print('модель NMF:')
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i+1}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

модель NMF:
Topic 1: ан заря, ан, заря, квадратный, квадратный метр, метр, район, отделка, ипотека, твой
Topic 2: курс, датасаентист, заказчик, навык, область, узконаправленный, хватить, дать, штука, выстроить
Topic 3: смартфон, оператор, немат, позиция, доллар, гигант, оборудование, оператор связь, технократия, аравинд
Topic 4: мина час назад, мина час, бейкер, онлайн, календарь, должность, февраль, февраль онлайн, февраль онлайн календарь, онлайн календарь
Topic 5: сообщество обзор перевод, песочница дата регистрация, адрес имя пароль, информация например адрес, представитель информация устройство, представитель информация, спасибо внимание корпоративный, похожий сайт ваш, похожий сайт, компания песочница дата


In [34]:
df[df['cluster']==2]

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
10,Как создать аппаратный эмулятор CD-ROM без пая...,"2394.92 Рейтинг Несмотря на то, что постепе...",20 мар в 14:00,"Хабы: Блог компании , Системное администриров...",2394.92,рейтинг несмотря на то что постепенно оптичес...,несмотря постепенно оптические диски уходят пр...,несмотр постепен оптическ диск уход прошл обра...,несмотря постепенно оптический диск уходить пр...,2


In [35]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==2]['text_lemm'])
print('модель NMF:')
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i+1}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

модель NMF:
Topic 1: устройство, который, образ, система, команда, эмулятор, диск, режим, компьютер, работать
Topic 2: устроенный буфер, свой, подробно, мочь работать, настройка, играть, интерес, система помощь команда, соединение, режим хост
Topic 3: наш, перекомпиляция, канал, встраивать устройство, карта вставлять её, широко, осветить интересность, ядро отсутствовать, процессор также, громоздкость решать проблема
Topic 4: несмотря постепенно, поддержка манипулятор мышь, устройство который доступный, ядро изменение несколько, общение использоваться, система подключиться, очень различный, необходимо знать команда, доступный обнаружение находить, запускать процесс процесс
Topic 5: аккаунт войти, создание устройство, понять делать, канал второй, который слушать, условие, ассоциировать жёсткий, регистрация март дата, вшить, стать пользоваться


In [36]:
df[df['cluster']==3]

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
4,Быстрое начало работы с Gitlab CI/CD: пайплайн...,4.29 Оценка 280.79 Рейтинг Southbridge Обеспе...,2 часа назад,"Хабы: Блог компании Southbridge , Тестирован...",280.79,оценка рейтинг обеспечиваем стабильную работ...,оценка обеспечиваем стабильную работу проектов...,оценк обеспечива стабильн работ проект оригина...,оценка обеспечивать стабильный оригинал переве...,3


In [37]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==3]['text_lemm'])
print('модель NMF:')
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i+1}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

модель NMF:
Topic 1: пайплайна, этап, который, команда, несколько, задание, сайт, компания, разработчик, настройка
Topic 2: февраль численность человек, уменьшение количество бутылочный, команда каждый который, тестирование невозможно, шаг позволять гарантировать, основный ветка, дата регистрация ноябрь, процесс который происходить, непрерывный, ноябрь дата основание
Topic 3: задание задание, сайт компания, автоматический, инструмент, некоторый, повышение, системный администрирование туториал, разработка уменьшение количество, корпоративный медийный реклама, создание пайплайна развёртывание
Topic 4: непрерывный интеграция, системный администрирование, интеграция, статический веб, несколько раннер, шаг, сборка, этап, случай, способность человек стимулировать
Topic 5: переполненный ами, полный список смотреть, преимущество бизнес, сотрудник, изменение автоматически развёртываться, пайплайна обычно, постоянно готовый деплоить, предположим хотеть простой, исходный, кодовый постоянно готовы

In [38]:
df[df['cluster']==4]

,title,description,date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
9,Как системному аналитику написать хорошее резю...,"look, use the source! 1. Указывайте количеств...",20 мар в 14:20,"Хабы: Анализ и проектирование систем, Карьера...",0.0,указывайте количественно и качественно вы...,указывайте количественно качественно выраженны...,указыва количествен качествен выражен достижен...,указывать количественно качественно выраженный...,4


In [39]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==4]['text_lemm'])
print('модель NMF:')
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i+1}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

модель NMF:
Topic 1: аналитик, резюме, системный, системный аналитик, указывать, писать, человек, реклама, который, слово
Topic 2: указывать место, хабра событие, новость, человек подчинение разница, мина робот, ошибка документ который, который представлять весь, вриокомуно никто, сказать, разбирать
Topic 3: вриокомуно, искать, компания зачастую говорить, робот марвин ждать, реклама пункт, работать ведущий, дно, россия реклама байка, допускать опечатка, туториал исследовать
Topic 4: ама байка, текст всё резать, публикация час назад, малое бизнес далее, рекомендация анализ, дело дойти показать, погромист сказать, хороший резюме рекомендация, образец документ запрос, войти регистрация раздел
Topic 5: обнаружить резюме человек, отдельный вопрос, подчинение разница, сайт сайт, выглядеть свидетель фрязино, человек который катастрофически, некоторый умудряться нагородить, писать беспомощный, подкреплять софт скилла, дать существенный эффект


## Классификация

In [40]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text_lemm'], df['cluster'], 
                                                      test_size=0.3, 
                                                      random_state=0)

In [41]:
len(X_train)

10

In [42]:
len(X_test)

5

In [43]:
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [44]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,3), stop_words=russian_stopwords)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## LogicRegression

In [45]:
model_lr = LogisticRegression()
model_lr.fit(X_train_tfidf, y_train)

LogisticRegression()

In [46]:
y_pred = model_lr.predict(X_test_tfidf)

In [47]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.60      1.00      0.75         3
           4       0.00      0.00      0.00         1

    accuracy                           0.60         5
   macro avg       0.20      0.33      0.25         5
weighted avg       0.36      0.60      0.45         5



In [48]:
X_test_tfidf.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [49]:
y_pred

array([1, 1, 1, 1, 1], dtype=int64)

## RandomForestClassifier

In [50]:
model_rf = RandomForestClassifier()
model_rf.fit(X_train_tfidf, y_train)

RandomForestClassifier()

In [51]:
y_pred = model_rf.predict(X_test_tfidf)

In [52]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.60      1.00      0.75         3
           4       0.00      0.00      0.00         1

    accuracy                           0.60         5
   macro avg       0.20      0.33      0.25         5
weighted avg       0.36      0.60      0.45         5



## KNeighborsClassifier

In [53]:
model_knn = KNeighborsClassifier()
model_knn.fit(X_train_tfidf, y_train)

KNeighborsClassifier()

In [54]:
y_pred = model_knn.predict(X_test_tfidf)

In [55]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.60      1.00      0.75         3
           4       0.00      0.00      0.00         1

    accuracy                           0.60         5
   macro avg       0.20      0.33      0.25         5
weighted avg       0.36      0.60      0.45         5



## Сохранение модели

In [56]:
import pickle

with open('habr_model.pkl', 'wb') as f:
    pickle.dump(model_knn, f)

In [57]:
with open('habr_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

In [58]:
df.to_csv('data_movies.csv')

In [59]:
t1 = 'Представлена открытая лёгковесная библиотека gpu.cpp для проведения упрощённых низкоуровневых вычислений на GPU с помощью C++. Исходный код проекта опубликован на GitHub под лицензией Apache License 2.0.Технические цели проекта — лёгкий вес, быстрая итерация и простой шаблон.Разработчики gpu.cpp пояснили, что в проекте используется спецификация WebGPU. Решение позволяет добавлять код для выполнения на GPU в проекты C++, включая графические модули Nvidia, Intel, AMD. Один и тот же код C++ может работать на самых разных ноутбуках, рабочих станциях, мобильных устройствах или практически на любом оборудовании с поддержкой Vulkan, Metal или DirectX.'

In [60]:
with open('habr_model.pkl', 'rb') as file:
    model = pickle.load(file)

with open('habr_vectorizer.pkl', 'rb') as file:
    vectorizer = pickle.load(file)

In [61]:
def fun_punctuation_text(text):
    text = text.lower()
    text = ''.join([ch for ch in text if ch not in string.punctuation])
    text = ''.join([i if not i.isdigit() else '' for i in text])
    text = ''.join([i if i.isalpha() else ' ' for i in text])
    text = re.sub(r'\s+', ' ', text, flags=re.I)
    text = re.sub('[a-z]', '', text, flags=re.I)
    st = '❯\xa0'
    text = ''.join([ch if ch not in st else ' ' for ch in text])
    return text

In [62]:
def fun_lemmatizing_text(text):
    tokens = word_tokenize(text)
    res = list()
    for word in tokens:
        p = pymorphy3.MorphAnalyzer(lang='ru').parse(word)[0]
        res.append(p.normal_form)  
    text = " ".join(res)
    return text

In [63]:
def fun_tokenize(text):
    t = word_tokenize(text)
    tokens = [token for token in t if token not in russian_stopwords]
    text = " ".join(tokens)
    return text

In [64]:
t1 = fun_tokenize(fun_lemmatizing_text(fun_punctuation_text(t1)))

In [65]:
text_vectorized = vectorizer.transform([t1])

In [66]:
text_vectorized.toarray()

array([[0., 0., 0., ..., 0., 0., 0.]])

In [67]:
prediction = model.predict(text_vectorized)  
probabilities = model.predict_proba(text_vectorized)

In [68]:
print(f"Класс: {prediction[0]}")
print(f"Вероятности: {probabilities}")

Класс: 1
Вероятности: [[0.6 0.2 0.2]]
